In [18]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

<h2>Generating synthetic alaska data</h2>


In [2]:
def generate_structured_alaska_data(num_samples=2000, lookback=6, height=16, width=16, channels=5):
    print("Generating data..")
    X = np.random.rand(num_samples, lookback, height, width, channels).astype(np.float32) * 0.5
    y = np.zeros((num_samples, 1), dtype=np.float32)
    
    # creating ~10% actual fire events
    num_fires = int(num_samples * 0.10)
    fire_indices = np.random.choice(num_samples, num_fires, replace=False)
    
    for i in fire_indices:
        y[i] = 1.0
        # Inject a "hotspot"in the most recent time steps
        # E.g., a 4x4 grid of extreme weather (High Temp[2], High Wind[3], Low Moisture[1])
        start_h, start_w = np.random.randint(0, height-4), np.random.randint(0, width-4)
        
        for j in range(lookback - 2, lookback): # The last two months get severe
            X[i, j, start_h:start_h+4, start_w:start_w+4, 2] += 0.4 # Spike Temp
            X[i, j, start_h:start_h+4, start_w:start_w+4, 3] += 0.4 # Spike Wind
            X[i, j, start_h:start_h+4, start_w:start_w+4, 1] -= 0.3 # Drop Moisture
            
    
    X = np.clip(X, 0.0, 1.0)
    
    print(f"Dataset generated. Class Distribution: {int(np.sum(y))} Fires, {len(y) - int(np.sum(y))} Non-Fires")
    return X, y

In [3]:
class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.75, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        
        loss = -alpha_t * tf.pow(1 - p_t, self.gamma) * tf.math.log(p_t)
        return tf.reduce_mean(loss)

<h2>HYBRID-CNN TRANSFORMER</h2>

In [20]:
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoderBlock, self).__init__(**kwargs)
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_hybrid_fire_model(lookback, height, width, channels):
    inputs = layers.Input(shape=(lookback, height, width, channels))
    
    # Added BatchNormalization to stabilize CNN learning
    x = layers.TimeDistributed(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)
    
    x = layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)
    
    embed_dim = 32
    positions = tf.range(start=0, limit=lookback, delta=1)
    pos_embeddings = layers.Embedding(input_dim=lookback, output_dim=embed_dim)(positions)
    x = x + pos_embeddings
    
    x = TransformerEncoderBlock(embed_dim=embed_dim, num_heads=4, ff_dim=64)(x)
    x = layers.Lambda(lambda tensor: tensor[:, -1, :])(x)
    
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [21]:
LOOKBACK, GRID_SIZE, CHANNELS = 6, 16, 5
BATCH_SIZE, EPOCHS = 32, 15
    
X_full, y_full = generate_structured_alaska_data(num_samples=2000)
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

model = build_hybrid_fire_model(LOOKBACK, GRID_SIZE, GRID_SIZE, CHANNELS)

Generating data..
Dataset generated. Class Distribution: 200 Fires, 1800 Non-Fires


In [23]:
early_stopping = EarlyStopping(
    monitor='val_pr_auc',  
    mode='max',            
    patience=5,            
    restore_best_weights=True
)
model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=BinaryFocalLoss(gamma=2.0, alpha=0.75), 
        metrics=[tf.keras.metrics.AUC(curve='PR', name='pr_auc'), 'accuracy']
    )
    
print("\nTraining with Focal Loss...")
model.fit(
    X_train, y_train, 
    batch_size=BATCH_SIZE, 
    epochs=50,             
    validation_split=0.2, 
    callbacks=[early_stopping],
    verbose=1
)    
print("\n--- Final Model Evaluation (Threshold Tuning) ---")
predictions_prob = model.predict(X_test)
    


Training with Focal Loss...
Epoch 1/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 17s 107ms/step - accuracy: 0.9305 - loss: 0.0188 - pr_auc: 0.7850 - val_accuracy: 0.9000 - val_loss: 0.2819 - val_pr_auc: 0.5749
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 0.9953 - loss: 0.0021 - pr_auc: 0.9968 - val_accuracy: 0.9000 - val_loss: 0.3125 - val_pr_auc: 0.8368
Epoch 3/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.9992 - loss: 7.1588e-04 - pr_auc: 0.9996 - val_accuracy: 1.0000 - val_loss: 8.9559e-05 - val_pr_auc: 1.0000
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 1.0000 - loss: 1.7955e-04 - pr_auc: 1.0000 - val_accuracy: 0.9937 - val_loss: 3.7000e-04 - val_pr_auc: 1.0000
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.9969 - loss: 9.6154e-04 - pr_auc: 0.9976 - val_accuracy: 0.5656 - val_loss: 0.0733 - val_pr_auc: 1.0000
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.9984 - loss: 0.0013 - pr_auc: 0.9989 - val_accuracy: 0.9781

In [24]:
thresholds = [0.3, 0.5, 0.7]
for thresh in thresholds:
    predictions_binary = (predictions_prob > thresh).astype(int)
    precision = precision_score(y_test, predictions_binary, zero_division=0)
    recall = recall_score(y_test, predictions_binary, zero_division=0)
    f1 = f1_score(y_test, predictions_binary, zero_division=0)
        
    print(f"\nThreshold: {thresh}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
        
pr_auc = average_precision_score(y_test, predictions_prob)
print(f"\nOverall PR-AUC: {pr_auc:.4f}")


Threshold: 0.3
  Precision: 1.0000
  Recall:    0.9500
  F1-Score:  0.9744

Threshold: 0.5
  Precision: 1.0000
  Recall:    0.9500
  F1-Score:  0.9744

Threshold: 0.7
  Precision: 1.0000
  Recall:    0.9250
  F1-Score:  0.9610

Overall PR-AUC: 1.0000


<h2>Conclusion</h2>

In our data generator, we created a very strict mathematical rule for fires: If there is a 4x4 pixel block of high heat, high wind, and low moisture in the last two months, it is a Fire.

While real-world Alaskan satellite data will never give you a 1.0 PR-AUC (because real weather is chaotic sensors have noise, and clouds block optical data).

this prototype proved three critical things:

The Architecture is Sound: The (T, C, H, W) tensor pipeline works. The TimeDistributed wrapper correctly passed spatial features to the Transformer.

The Custom Focal Loss Works: It completely solved the "Probability Squashing" issue from the first run.